In [10]:
import nflreadpy as nfl
import pathlib
import polars as pl
from polars import dtype_of

In [11]:
#Shared variables

# seasons to retrieve data for
seasons_to_retrieve = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

In [12]:
# Player data

# player positions
all_positions = ['P', 'TE', 'OT', 'QB', 'FB', 'S', 'C', 'CB', 'DL', 'WR', 'MLB', 'OLB', 'DE', None, 'DB', 'LS', 'NT',
                 'SAF', 'G', 'DT', 'ILB', 'FS', 'K', 'RB', 'LB']

# player positions needed for fantasy scoring metrics
fantasy_relevant_positions = ['TE', 'QB', 'WR', 'K', 'RB']

# fantasy scoring fields (with a few extra for info)
fantasy_cols = [
    # Player identification
    'player_id', 'player_display_name', 'position', 'recent_team', 'season',

    # Passing (QB)
    'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'passing_2pt_conversions',

    # Rushing (RB/QB)
    'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost', 'rushing_2pt_conversions',

    # Receiving (WR/RB/TE) - PPR CRITICAL
    'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles_lost',
    'receiving_2pt_conversions',

    # Kicking
    'fg_made', 'fg_missed', 'pat_made', 'fg_made_0_19', 'fg_made_20_29', 'fg_made_30_39', 'fg_made_40_49',
    'fg_made_50_59', 'fg_made_60_',

    # ALREADY CALCULATED (use these directly)
    'fantasy_points', 'fantasy_points_ppr'
]

# retrieval choices
positions_to_retrieve = fantasy_relevant_positions
cols_to_retrieve = fantasy_cols

# create a query to get the relevant data
get_relevant_player_data = (nfl.load_player_stats(
    seasons=seasons_to_retrieve,
    summary_level="reg"
)[cols_to_retrieve].lazy()
                            .filter(pl.col('position').is_in(positions_to_retrieve))
                            .sort(['season', 'position'])
                            )

player_data = get_relevant_player_data.collect()

In [13]:
# Team Data

# seasons to retrieve data for
seasons_to_retrieve = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# scoring categories for team data (defense and special teams in this case)
fantasy_dst_cols = [
    # IDENTIFICATION
    'season', 'team', 'games',

    # SACKS
    'def_sacks',

    # TURNOVERS
    'def_interceptions', 'def_fumbles_forced', 'fumble_recovery_opp',

    # TDs (Defense/Special Teams)
    'def_tds', 'special_teams_tds', 'fumble_recovery_tds',

    # SAFETIES
    'def_safeties',
]

# retrieval choices
cols_to_retrieve = fantasy_dst_cols

# retrieve and export team data for chosen seasons
get_relevant_team_data = (
    nfl.load_team_stats(
        seasons=seasons_to_retrieve,
        summary_level="reg"
    )[cols_to_retrieve].lazy()
    .sort(['season'])
)

team_data = get_relevant_team_data.collect()

In [14]:
# get files as dataframes
excel_files_to_combine = []
csv_files_to_combine = []

frames = [player_data, team_data]

for file in excel_files_to_combine:
    frames.append(pl.read_excel(file))

for file in csv_files_to_combine:
    frames.append(pl.read_csv(file))

frames = [
    frame.rename({'recent_team': 'team'})
    if 'recent_team' in frame.columns
    else frame
    for frame in frames
]

In [15]:
# combine files
combined_frame = (pl.concat(frames, how='diagonal_relaxed')
                  .fill_null(strategy='zero'))

print(f"nulls: {combined_frame.null_count().sum_horizontal()}")

for column in combined_frame.columns:
    print(f"Column: {column} | Dtype: {combined_frame[column].dtype}")

nulls: shape: (1,)
Series: 'sum' [u32]
[
	0
]
Column: player_id | Dtype: String
Column: player_display_name | Dtype: String
Column: position | Dtype: String
Column: team | Dtype: String
Column: season | Dtype: Int32
Column: completions | Dtype: Int32
Column: attempts | Dtype: Int32
Column: passing_yards | Dtype: Int32
Column: passing_tds | Dtype: Int32
Column: passing_interceptions | Dtype: Int32
Column: passing_2pt_conversions | Dtype: Int32
Column: carries | Dtype: Int32
Column: rushing_yards | Dtype: Int32
Column: rushing_tds | Dtype: Int32
Column: rushing_fumbles_lost | Dtype: Int32
Column: rushing_2pt_conversions | Dtype: Int32
Column: receptions | Dtype: Int32
Column: targets | Dtype: Int32
Column: receiving_yards | Dtype: Int32
Column: receiving_tds | Dtype: Int32
Column: receiving_fumbles_lost | Dtype: Int32
Column: receiving_2pt_conversions | Dtype: Int32
Column: fg_made | Dtype: Int32
Column: fg_missed | Dtype: Int32
Column: pat_made | Dtype: Int32
Column: fg_made_0_19 | Dtyp

In [16]:
# prepare output
year_begin = seasons_to_retrieve[0]
year_end = seasons_to_retrieve[-1]

path = pathlib.Path.cwd() / f"combined_fantasy_data_{year_begin}_{year_end}"

In [17]:
# output combined file as csv
print(combined_frame.write_csv(str(path)+".csv", include_header=True))

None


In [18]:
# output combined file as parquet
print(combined_frame.write_parquet(str(path)+".parquet"))

None
